# Training a Diffusion Model (DDPM) on MNIST with Kubeflow Trainer

This example demonstrates how to train a **Denoising Diffusion Probabilistic Model (DDPM)** to generate handwritten digit images using the [MNIST](http://yann.lecun.com/exdb/mnist/) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html).

## What is a Diffusion Model?

Diffusion models generate images through a two-phase process:

1. **Forward process (adding noise)**: Gradually add Gaussian noise to a real image over many timesteps until it becomes pure noise. This is fixed (not learned) and follows a predefined schedule.

2. **Reverse process (denoising)**: Train a neural network to reverse the noise process -- given a noisy image and the timestep, predict the noise that was added. At generation time, start from pure random noise and iteratively denoise to produce a realistic image.

**DDPM** (Ho et al., 2020) is the foundational algorithm that made this practical. The model learns to predict the noise component at each timestep using a simple MSE loss. This is fundamentally different from classification -- there are no labels, and the loss measures how well the model predicts noise.

This notebook trains a small UNet model (~500K parameters) entirely from scratch. With 3 epochs on CPU, you will see the loss decrease and the model begin to learn digit-like structures. For clean generated images, more epochs on GPU are recommended.

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to be able to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Define the Training Function

The training function contains everything needed to train a DDPM diffusion model:

- **Noise schedule**: Linear beta schedule that controls how much noise is added at each timestep
- **Forward diffusion**: Closed-form formula to jump to any noise level in one step
- **UNet architecture**: A small neural network with skip connections and timestep conditioning that predicts the noise added to an image
- **Training loop**: Sample random timesteps, add noise, predict the noise, minimize MSE loss
- **Sampling**: After training, generate new images by starting from pure noise and iteratively denoising

All code is defined inside a single function because Kubeflow Trainer serializes and ships this function to distributed worker nodes.

In [ ]:
def train_ddpm_mnist():
    import math
    import os

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # ── Hyperparameters ──────────────────────────────────────────────────
    num_timesteps = 100
    num_epochs = 3
    batch_size = 64
    learning_rate = 1e-3

    # ── Noise Schedule ───────────────────────────────────────────────────
    # Linear beta schedule: small noise at t=0, larger noise at t=T
    betas = torch.linspace(1e-4, 0.02, num_timesteps)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

    # ── Forward Diffusion ────────────────────────────────────────────────
    def q_sample(x_start, t, noise):
        """Add noise to x_start at timestep t using the closed-form formula:
        x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        """
        sqrt_alpha = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1).to(x_start.device)
        sqrt_one_minus_alpha = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1).to(x_start.device)
        return sqrt_alpha * x_start + sqrt_one_minus_alpha * noise

    # ── Timestep Embedding ───────────────────────────────────────────────
    class SinusoidalEmbedding(nn.Module):
        """Sinusoidal positional encoding for timesteps.
        Same concept as transformer positional encodings -- maps integer
        timesteps to continuous vectors using sin/cos at different frequencies.
        """
        def __init__(self, dim):
            super().__init__()
            self.dim = dim

        def forward(self, t):
            device = t.device
            half_dim = self.dim // 2
            emb = math.log(10000) / (half_dim - 1)
            emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
            emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
            emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
            return emb

    # ── UNet Building Blocks ─────────────────────────────────────────────
    class ResidualBlock(nn.Module):
        """Residual block with timestep conditioning.
        Conv -> GroupNorm -> SiLU -> Conv -> GroupNorm -> SiLU + skip connection.
        Timestep embedding is added after the first convolution.
        """
        def __init__(self, in_channels, out_channels, time_dim):
            super().__init__()
            self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
            self.norm1 = nn.GroupNorm(8, out_channels)
            self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
            self.norm2 = nn.GroupNorm(8, out_channels)
            self.time_mlp = nn.Linear(time_dim, out_channels)
            self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

        def forward(self, x, t_emb):
            h = F.silu(self.norm1(self.conv1(x)))
            # Add timestep conditioning
            h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
            h = F.silu(self.norm2(self.conv2(h)))
            return h + self.skip(x)

    class SimpleUNet(nn.Module):
        """Minimal UNet for 28x28 grayscale images.

        Architecture:
          Encoder:  1 -> 32 channels (28x28) -> MaxPool -> 32 -> 64 channels (14x14) -> MaxPool (7x7)
          Bottleneck: 64 channels at 7x7
          Decoder:  Upsample -> concat skip -> 128 -> 64 (14x14) -> Upsample -> concat skip -> 96 -> 32 (28x28)
          Output:   Conv2d(32, 1, 1)

        The model predicts the noise that was added to the input image.
        """
        def __init__(self, time_dim=64):
            super().__init__()
            # Timestep embedding: integer -> vector
            self.time_embed = nn.Sequential(
                SinusoidalEmbedding(time_dim),
                nn.Linear(time_dim, time_dim),
                nn.SiLU(),
            )

            # Encoder
            self.down1 = ResidualBlock(1, 32, time_dim)      # 28x28
            self.down2 = ResidualBlock(32, 64, time_dim)      # 14x14
            self.pool = nn.MaxPool2d(2)

            # Bottleneck at 7x7
            self.bottleneck = ResidualBlock(64, 64, time_dim)

            # Decoder (channels after concat: 64+64=128, then 32+64=96)
            self.up2 = ResidualBlock(128, 64, time_dim)       # 14x14
            self.up1 = ResidualBlock(96, 32, time_dim)        # 28x28

            self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
            self.out_conv = nn.Conv2d(32, 1, 1)

        def forward(self, x, t):
            t_emb = self.time_embed(t)

            # Encoder path
            d1 = self.down1(x, t_emb)              # (B, 32, 28, 28)
            d2 = self.down2(self.pool(d1), t_emb)   # (B, 64, 14, 14)
            b = self.bottleneck(self.pool(d2), t_emb) # (B, 64, 7, 7)

            # Decoder path with skip connections
            up2 = self.upsample(b)                   # (B, 64, 14, 14)
            up2 = self.up2(torch.cat([up2, d2], dim=1), t_emb)  # (B, 64, 14, 14)
            up1 = self.upsample(up2)                 # (B, 64, 28, 28)
            up1 = self.up1(torch.cat([up1, d1], dim=1), t_emb)  # (B, 32, 28, 28)

            return self.out_conv(up1)                # (B, 1, 28, 28)

    # ── Distributed Setup ────────────────────────────────────────────────
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    device = torch.device(f"{device}:{local_rank}")

    # Move schedule tensors to device
    sqrt_alphas_cumprod_dev = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alphas_cumprod_dev = sqrt_one_minus_alphas_cumprod.to(device)

    # ── Model ────────────────────────────────────────────────────────
    model = nn.parallel.DistributedDataParallel(SimpleUNet().to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    param_count = sum(p.numel() for p in model.parameters())
    if dist.get_rank() == 0:
        print(f"Model parameters: {param_count:,}")

    # ── Data ─────────────────────────────────────────────────────────
    # Normalize images to [-1, 1] -- this is standard for diffusion models
    # so that the noise (standard normal) and the data are on the same scale.
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    # Download dataset only on local_rank=0 to avoid race conditions.
    if local_rank == 0:
        dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
    dist.barrier()
    dataset = datasets.MNIST("./data", train=True, download=False, transform=transform)

    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=DistributedSampler(dataset),
    )

    # ── Training Loop ────────────────────────────────────────────────────
    dist.barrier()
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loader.sampler.set_epoch(epoch)

        for batch_idx, (images, _labels) in enumerate(train_loader):
            # Note: _labels are ignored -- diffusion is unsupervised
            images = images.to(device)

            # Sample random timesteps for each image in the batch
            t = torch.randint(0, num_timesteps, (images.size(0),), device=device)

            # Sample noise and create noisy images
            noise = torch.randn_like(images)
            noisy_images = q_sample(images, t, noise)

            # Predict the noise that was added
            predicted_noise = model(noisy_images, t)

            # MSE loss between predicted and actual noise
            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_idx % 100 == 0 and dist.get_rank() == 0:
                print(
                    "Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(images),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

    # ── Sampling (rank 0 only) ───────────────────────────────────────────
    dist.barrier()
    if dist.get_rank() == 0:
        print("\nGenerating sample images...")
        model.eval()

        # Precompute coefficients for the reverse process
        betas_dev = betas.to(device)
        alphas_dev = alphas.to(device)
        alphas_cumprod_dev = alphas_cumprod.to(device)

        with torch.no_grad():
            # Start from pure Gaussian noise
            x = torch.randn(16, 1, 28, 28, device=device)

            # Reverse diffusion: denoise step by step from t=T-1 to t=0
            for t_step in reversed(range(num_timesteps)):
                t_batch = torch.full((16,), t_step, device=device, dtype=torch.long)

                # Predict noise at this timestep
                pred_noise = model(x, t_batch)

                # DDPM reverse step formula
                alpha_t = alphas_dev[t_step]
                alpha_bar_t = alphas_cumprod_dev[t_step]
                beta_t = betas_dev[t_step]

                # Predicted mean: (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * predicted_noise)
                mean = (1.0 / torch.sqrt(alpha_t)) * (
                    x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * pred_noise
                )

                # Add noise for all steps except the last one (t=0)
                if t_step > 0:
                    noise = torch.randn_like(x)
                    sigma = torch.sqrt(beta_t)
                    x = mean + sigma * noise
                else:
                    x = mean

            # Print statistics of generated samples (we cannot display images in a container)
            print(f"Generated {x.shape[0]} images of shape {x.shape[1:]}")
            print(f"Pixel value range: [{x.min().item():.3f}, {x.max().item():.3f}]")
            print(f"Pixel mean: {x.mean().item():.3f}, std: {x.std().item():.3f}")
            print("Note: With only 3 epochs, generated images will be blurry.")
            print("Train for 10+ epochs on GPU for clearer digit generation.")

    # ── Cleanup ──────────────────────────────────────────────────────
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    dist.destroy_process_group()

## Run the Training Locally

We can submit the training function to the local Trainer client to run it in an isolated subprocess.

This will train the DDPM model on a single process. On CPU, expect ~10-15 minutes for 3 epochs. The loss should decrease from ~0.5 to ~0.1-0.2, indicating the model is learning to predict noise.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_ddpm_mnist,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale DDPM Training with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

Distributed training is especially useful for diffusion models because:
- Each node processes a different subset of the dataset (data parallelism)
- Gradient averaging across nodes produces more stable training
- Training time decreases roughly linearly with the number of nodes

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

Kubeflow TrainJob will train the DDPM model on 2 PyTorch nodes. Each node gets its own shard of the MNIST dataset.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_ddpm_mnist,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=2,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 2,
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)

## Check the TrainJob Steps

You can check the components of TrainJob that have been created.

Since the TrainJob performs distributed training across 2 nodes, it generates 2 steps: `trainer-node-0` and `trainer-node-1`.

In [ ]:
# Wait for the running status.
client.wait_for_job_status(name=job_name, status={"Running"})

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

Since we run training on 2 nodes, each node processes 60,000/2 = 30,000 images per epoch.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.

In [ ]:
# client.delete_job(job_name)